# 02 - Data Profiling and Quality Review

Este notebook resume resultados locales de profiling y calidad para Landing, Bronze y Silver. No transforma datos, no ejecuta Spark pesado y no exporta CSV, HTML ni otros artefactos derivados.

Los archivos JSONL de calidad son locales y pueden no existir en todos los entornos. Si faltan, el notebook muestra un mensaje controlado y contin?a.


## Preparaci?n del entorno

Se resuelve `PROJECT_ROOT` para que el notebook funcione ejecutado desde la ra?z del repositorio o desde la carpeta `notebooks`.


In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

QUALITY_DIR = PROJECT_ROOT / 'data' / 'quality'
BRONZE_RESULTS_PATH = QUALITY_DIR / 'bronze_quality_results.jsonl'
SILVER_RESULTS_PATH = QUALITY_DIR / 'silver_quality_results.jsonl'

PROJECT_ROOT


## Carga de resultados

Se leen los JSONL de calidad Bronze y Silver si existen. No se escriben archivos desde el notebook.


In [ ]:
def read_quality_jsonl(path: Path, layer: str) -> list[dict]:
    """Lee resultados JSONL de calidad si existen."""

    if not path.exists():
        print(f'No existe {path}. Ejecuta primero el motor de calidad de {layer}.')
        return []

    with path.open('r', encoding='utf-8') as file:
        rows = [json.loads(line) for line in file if line.strip()]

    for row in rows:
        row['layer'] = layer
        if 'rule_name' not in row and 'rule_id' in row:
            row['rule_name'] = row['rule_id']

    return rows

quality_rows = [
    *read_quality_jsonl(BRONZE_RESULTS_PATH, layer='bronze'),
    *read_quality_jsonl(SILVER_RESULTS_PATH, layer='silver'),
]

quality_df = pd.DataFrame(quality_rows)
quality_df.head()


## Resumen general

Vista agregada por capa con total de resultados y estados `PASS`, `WARNING` y `FAIL`.


In [ ]:
def status_summary(df: pd.DataFrame, group_columns: list[str]) -> pd.DataFrame:
    """Construye una tabla de conteos PASS/WARNING/FAIL."""

    if df.empty:
        return pd.DataFrame(columns=[*group_columns, 'total_results', 'PASS', 'WARNING', 'FAIL'])

    summary = (
        df.groupby([*group_columns, 'status'])
        .size()
        .unstack(fill_value=0)
        .reset_index()
    )

    for status in ['PASS', 'WARNING', 'FAIL']:
        if status not in summary.columns:
            summary[status] = 0

    summary['total_results'] = summary[['PASS', 'WARNING', 'FAIL']].sum(axis=1)
    return summary[[*group_columns, 'total_results', 'PASS', 'WARNING', 'FAIL']]

general_summary_df = status_summary(quality_df, ['layer'])
general_summary_df


## Resumen por fuente


In [ ]:
source_summary_df = status_summary(quality_df, ['layer', 'source_name'])
source_summary_df


## Resumen por regla


In [ ]:
rule_summary_df = status_summary(quality_df, ['layer', 'rule_name'])
rule_summary_df.sort_values(['layer', 'WARNING', 'FAIL', 'rule_name'], ascending=[True, False, False, True])


## Vista de nulos cr?ticos

Incluye reglas de nulos cr?ticos en Silver y columnas completamente nulas en Bronze.


In [ ]:
null_rules = {'critical_nulls', 'fully_null_columns'}
null_quality_df = quality_df[quality_df.get('rule_name', pd.Series(dtype=str)).isin(null_rules)] if not quality_df.empty else quality_df
null_quality_df[['layer', 'source_name', 'resource_key', 'rule_name', 'status', 'message']].head(50) if not null_quality_df.empty else null_quality_df


## Vista de duplicados

Los duplicados por llave candidata no prueban error de datos; pueden indicar que la granularidad real requiere m?s columnas.


In [ ]:
duplicate_rules = {
    'exact_duplicate_rows',
    'candidate_key_duplicates',
    'siaf_candidate_key_duplicates',
    'sismepre_candidate_key_duplicates',
    'renamu_ubigeo_duplicates',
    'renamu_idmunici_duplicates',
}
duplicate_quality_df = quality_df[quality_df.get('rule_name', pd.Series(dtype=str)).isin(duplicate_rules)] if not quality_df.empty else quality_df
duplicate_quality_df[['layer', 'source_name', 'resource_key', 'rule_name', 'status', 'message']].head(80) if not duplicate_quality_df.empty else duplicate_quality_df


## Vista de calidad num?rica


In [ ]:
numeric_rules = {'negative_amounts', 'predial_parse_failures', 'renamu_financial_parse_failures'}
numeric_quality_df = quality_df[quality_df.get('rule_name', pd.Series(dtype=str)).isin(numeric_rules)] if not quality_df.empty else quality_df
numeric_quality_df[['layer', 'source_name', 'resource_key', 'rule_name', 'status', 'message']].head(80) if not numeric_quality_df.empty else numeric_quality_df


## Vista de validaciones territoriales


In [ ]:
territory_pattern = 'ubigeo|tipomuni|territory|territorial'
territory_quality_df = quality_df[quality_df.get('rule_name', pd.Series(dtype=str)).str.contains(territory_pattern, case=False, na=False)] if not quality_df.empty else quality_df
territory_quality_df[['layer', 'source_name', 'resource_key', 'rule_name', 'status', 'message']].head(80) if not territory_quality_df.empty else territory_quality_df


## Interpretaci?n

- No hubo `FAIL` en Silver en la corrida local observada.
- Los `WARNING` no bloquean por s? solos, pero informan riesgos antes de integrar.
- Los duplicados por llave candidata no significan necesariamente filas err?neas; pueden indicar granularidad m?s fina.
- Los montos negativos requieren interpretaci?n presupuestal o contable antes de usarse en m?tricas finales.
- Los parseos fallidos puntuales deben revisarse antes de Gold.
- La integraci?n debe considerar estos hallazgos para refinar llaves, campos cr?ticos y reglas de consistencia territorial.
